# 1. Import Libraries

In [1]:
import numpy as np
import os

from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import (Embedding, LSTM, Dense, Dropout)
from tensorflow.keras.optimizers import Adam
from tensorflow.keras.callbacks import (EarlyStopping, ModelCheckpoint)

c:\Users\HP-PC\anaconda3\envs\myenv\Lib\site-packages\requests\__init__.py:86: RequestsDependencyWarning: Unable to find acceptable character detection dependency (chardet or charset_normalizer).
  warnings.warn(


# 2. Load Feature Engineering Outputs

In [2]:
processed_path = "../data/processed"

X_train = np.load(f"{processed_path}/X_train.npy")
X_test = np.load(f"{processed_path}/X_test.npy")

y_train = np.load(f"{processed_path}/y_train.npy")
y_test = np.load(f"{processed_path}/y_test.npy")

print("Training data:")
print(X_train.shape)

print("\nTesting data:")
print(X_test.shape)

print("\nLabels:")
print(y_train.shape)

Training data:
(35744, 300)

Testing data:
(8936, 300)

Labels:
(35744,)


# 3. Build LSTM Model

In [ ]:
# Configuration
MAX_WORDS = 20000
MAX_LENGTH = 300

model = Sequential()

# Embedding layer
model.add(Embedding(input_dim=MAX_WORDS, output_dim=128, input_length=MAX_LENGTH))

# LSTM layer
model.add(LSTM(units=128, return_sequences=False))
model.add(Dropout(0.5)) # Dropout to reduce overfitting
model.add(Dense(1, activation="sigmoid")) # Binary classification output

model.summary()

c:\Users\HP-PC\anaconda3\envs\myenv\Lib\site-packages\keras\src\layers\core\embedding.py:97: UserWarning: Argument `input_length` is deprecated. Just remove it.
  warnings.warn(


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ embedding (Embedding)           │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ lstm (LSTM)                     │ ?                      │   0 (unbuilt) │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout (Dropout)               │ ?                      │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ ?                      │   0 (unbuilt) │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 0 (0.00 B)

 Trainable params: 0 (0.00 B)

 Non-trainable params: 0 (0.00 B)

# 4. Compile Model

In [4]:
model.compile(
    optimizer=Adam(
        learning_rate=0.001
    ),
    loss="binary_crossentropy",
    metrics=[
        "accuracy"
    ]
)

# 5. Training Model

In [5]:
# Training Callbacks
os.makedirs(
    "../models",
    exist_ok=True
)

early_stopping = EarlyStopping(
    monitor="val_loss",
    patience=3,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    filepath=MODEL_PATH,
    monitor="val_accuracy",
    save_best_only=True,
    mode="max"
)

In [6]:
# Model Training
history = model.fit(
    X_train,
    y_train,
    validation_split=0.2,
    epochs=10,
    batch_size=64,
    callbacks=[
        early_stopping,
        checkpoint
    ]
)

Epoch 1/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 108s 237ms/step - accuracy: 0.7330 - loss: 0.5342 - val_accuracy: 0.9131 - val_loss: 0.2861
Epoch 2/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 112s 251ms/step - accuracy: 0.9267 - loss: 0.2551 - val_accuracy: 0.9143 - val_loss: 0.2544
Epoch 3/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 114s 255ms/step - accuracy: 0.9184 - loss: 0.2544 - val_accuracy: 0.8330 - val_loss: 0.5388
Epoch 4/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 134s 299ms/step - accuracy: 0.8852 - loss: 0.3208 - val_accuracy: 0.7786 - val_loss: 0.4809
Epoch 5/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 135s 302ms/step - accuracy: 0.9282 - loss: 0.2107 - val_accuracy: 0.9782 - val_loss: 0.0801
Epoch 6/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 115s 256ms/step - accuracy: 0.9703 - loss: 0.0888 - val_accuracy: 0.9727 - val_loss: 0.0893
Epoch 7/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 111s 248ms/step - accuracy: 0.9903 - loss: 0.0474 - val_accuracy: 0.9712 - val_loss: 0.1136
Epoch 8/10
447/447 ━━━━━━━━━━━━━━━━━━━━ 112s 251ms/step - accuracy: 0.9800 -

# 6. Save Model

In [7]:
MODEL_PATH = "../models/lstm_v2.keras"
model.save(MODEL_PATH)

print("Model saved successfully:")
print(MODEL_PATH)

Model saved successfully:
../models/lstm_v2.keras
